# 🥉 Bronze — `bronze.flights_raw`

Ingestão dos dados brutos, o mais próximo possível da origem.

|  |  |
|---|---|
| **Lê** | `/Volumes/workspace/default/dado_bruto_de_voos/flight_data_2024.csv` (~1,28 GB) |
| **Grava** | `bronze.flights_raw` (Delta Lake) |
| **Colunas** | 35 originais + 2 de rastreabilidade (`ingestion_timestamp`, `source_file`) |
| **Regido por** | `architecture.md` (camada Bronze) · `data_dictionary.md` (schema das 35 colunas) |

### Decisões desta camada

- **Schema explícito, sem `inferSchema=True`.** A inferência obriga o Spark a ler o arquivo duas vezes — caro em 7M+ linhas. O schema vem do dicionário de dados (Etapa 6).
- **Nenhuma regra de negócio aqui.** Sem deduplicação, sem filtro, sem agregação — isso é trabalho da Silver e da Gold. A Bronze só materializa o CSV em Delta e carimba de onde ele veio.
- **A ingestão é esta célula.** Não existe notebook separado de ingestão: ler o CSV e gravar a Bronze são o mesmo passo.

> **Execução com o dataset completo (Etapa 15):** 7.079.081 registros em 15,9s — Databricks Free Edition, Compute Serverless.


In [0]:
# ============================================================
# ETAPA 15 — EXECUÇÃO COM DATASET COMPLETO (FASE B)
# Mesmo pipeline validado no sample, agora com flight_data_2024.csv
# (7M+ registros)
# ============================================================
# CÉLULA 1 — Ingestão Bronze com schema explícito
# ============================================================

import time
from pyspark.sql import functions as F, types as T

inicio_execucao = time.time()

# Caminho do dataset completo
caminho_csv_completo = "/Volumes/workspace/default/dado_bruto_de_voos/flight_data_2024.csv"

# Schema explícito, baseado no docs/data_dictionary.md (Etapa 6)
# Evita inferSchema=True, que faz uma leitura extra e é mais lento em 7M+ linhas
schema = T.StructType([
    T.StructField("year", T.IntegerType(), True),
    T.StructField("month", T.IntegerType(), True),
    T.StructField("day_of_month", T.IntegerType(), True),
    T.StructField("day_of_week", T.IntegerType(), True),
    T.StructField("fl_date", T.DateType(), True),
    T.StructField("op_unique_carrier", T.StringType(), True),
    T.StructField("op_carrier_fl_num", T.DoubleType(), True),
    T.StructField("origin", T.StringType(), True),
    T.StructField("origin_city_name", T.StringType(), True),
    T.StructField("origin_state_nm", T.StringType(), True),
    T.StructField("dest", T.StringType(), True),
    T.StructField("dest_city_name", T.StringType(), True),
    T.StructField("dest_state_nm", T.StringType(), True),
    T.StructField("crs_dep_time", T.IntegerType(), True),
    T.StructField("dep_time", T.DoubleType(), True),
    T.StructField("dep_delay", T.DoubleType(), True),
    T.StructField("taxi_out", T.DoubleType(), True),
    T.StructField("wheels_off", T.DoubleType(), True),
    T.StructField("wheels_on", T.DoubleType(), True),
    T.StructField("taxi_in", T.DoubleType(), True),
    T.StructField("crs_arr_time", T.IntegerType(), True),
    T.StructField("arr_time", T.DoubleType(), True),
    T.StructField("arr_delay", T.DoubleType(), True),
    T.StructField("cancelled", T.IntegerType(), True),
    T.StructField("cancellation_code", T.StringType(), True),
    T.StructField("diverted", T.IntegerType(), True),
    T.StructField("crs_elapsed_time", T.DoubleType(), True),
    T.StructField("actual_elapsed_time", T.DoubleType(), True),
    T.StructField("air_time", T.DoubleType(), True),
    T.StructField("distance", T.DoubleType(), True),
    T.StructField("carrier_delay", T.IntegerType(), True),
    T.StructField("weather_delay", T.IntegerType(), True),
    T.StructField("nas_delay", T.IntegerType(), True),
    T.StructField("security_delay", T.IntegerType(), True),
    T.StructField("late_aircraft_delay", T.IntegerType(), True),
])

# 10.1 - Ler o CSV completo com schema definido
df_bronze = spark.read.csv(caminho_csv_completo, header=True, schema=schema)

# 10.4 - Adicionar metadados
df_bronze = df_bronze.withColumn("ingestion_timestamp", F.current_timestamp()) \
                      .withColumn("source_file", F.lit("flight_data_2024.csv"))

# 10.5 - Salvar como Delta (sobrescreve a versão do sample)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze.flights_raw")

contagem_bronze = df_bronze.count()
print("Registros salvos em bronze.flights_raw:", contagem_bronze)
print(f"Tempo até aqui: {round(time.time() - inicio_execucao, 1)}s")


Registros salvos em bronze.flights_raw: 7079081
Tempo até aqui: 15.9s
